# 3.5.7 Model Interpretation & Explainability

Permutation importance, Partial Dependence Plots, manual leave-one-out attribution.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings('ignore')
h=fetch_california_housing(); X,y=h.data,h.target
Xt,Xv,yt,yv=train_test_split(X,y,test_size=.2,random_state=42)
rf=RandomForestRegressor(100,random_state=42,n_jobs=-1); rf.fit(Xt,yt)
res=permutation_importance(rf,Xv,yv,n_repeats=10,random_state=42)
for i in res.importances_mean.argsort()[::-1]:
    print(f'{h.feature_names[i]:20s}: {res.importances_mean[i]:.4f} ± {res.importances_std[i]:.4f}')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import PartialDependenceDisplay
gbm=GradientBoostingRegressor(100,random_state=42); gbm.fit(Xt,yt)
top2=permutation_importance(gbm,Xv,yv,n_repeats=5,random_state=42).importances_mean.argsort()[-2:][::-1].tolist()
fig,ax=plt.subplots(1,2,figsize=(10,4))
PartialDependenceDisplay.from_estimator(gbm,Xt,top2,ax=ax,feature_names=h.feature_names)
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
p=make_pipeline(StandardScaler(),GradientBoostingRegressor(50,random_state=42)); p.fit(Xt,yt)
base=mean_squared_error(yv,p.predict(Xv))**.5; rng=np.random.default_rng(42)
print(f'Base RMSE={base:.4f}')
for i,name in enumerate(h.feature_names):
    Xp=Xv.copy(); Xp[:,i]=rng.permutation(Xp[:,i])
    print(f'{name:20s}: Δ RMSE={mean_squared_error(yv,p.predict(Xp))**.5-base:+.4f}')